In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path
import sys

# Add project root to path
ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

print("Libraries loaded successfully")
print(f"ChromaDB version : {chromadb.__version__}")

/Users/sidmohandidir/predictai-industrial/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully
ChromaDB version : 1.5.5


In [2]:
# Initialize ChromaDB — stores our document vectors locally
# persist_directory = where ChromaDB saves data on disk
chroma_client = chromadb.PersistentClient(
    path=str(ROOT / "data" / "chroma_db")
)

# Create a collection — like a table in a traditional database
# Each collection stores vectors for a specific topic
collection = chroma_client.get_or_create_collection(
    name="maintenance_docs",
    metadata={"description": "Industrial maintenance technical documentation"}
)

print(f"ChromaDB initialized")
print(f"Collection : maintenance_docs")
print(f"Documents stored : {collection.count()}")

ChromaDB initialized
Collection : maintenance_docs
Documents stored : 0


In [3]:
# Technical documents about industrial maintenance
# In a real project these would come from PDF manuals, ISO standards, etc.
# Here we create representative texts to demonstrate the RAG pipeline

documents = [
    {
        "id": "doc_001",
        "text": "High Pressure Compressor (HPC) degradation is one of the most common failure modes in turbofan engines. Symptoms include gradual decrease in HPC efficiency, rising exhaust gas temperature (EGT), and increased fuel consumption. When sensor readings show HPC efficiency below 47%, immediate inspection is recommended. Recommended action: borescope inspection of HPC blades stages 6-9, check for tip clearance issues and blade erosion.",
        "source": "Turbofan Engine Maintenance Manual — Chapter 7"
    },
    {
        "id": "doc_002", 
        "text": "Fan degradation in turbofan engines manifests as reduced fan speed (N1) and decreased bypass ratio. Physical fan speed readings above 9000 rpm combined with low corrected fan speed indicate fan efficiency loss. Common causes include foreign object damage (FOD), blade erosion, and seal deterioration. Inspection procedure: visual inspection of fan blades, measurement of blade tip clearance, check of fan case abradable coating.",
        "source": "Turbofan Engine Maintenance Manual — Chapter 3"
    },
    {
        "id": "doc_003",
        "text": "Remaining Useful Life (RUL) prediction is a key metric in predictive maintenance. When RUL falls below 30 cycles, the equipment enters WARNING status and maintenance planning should begin immediately. When RUL falls below 20 cycles, CRITICAL status is declared and equipment should be taken offline for inspection within 48 hours. Failure to act on CRITICAL alerts increases risk of unplanned shutdown by 73%.",
        "source": "Predictive Maintenance Best Practices — ISO 13374"
    },
    {
        "id": "doc_004",
        "text": "Thermal stress in turbine sections occurs when operating temperatures exceed design limits. High Pressure Turbine (HPT) nozzle guide vanes are particularly vulnerable. Indicators include elevated sensor readings for total temperature at HPC outlet (above 1600°C) and bleed enthalpy anomalies. Maintenance action: thermal barrier coating inspection, measurement of cooling hole diameters, replacement of vanes showing more than 15% area blockage.",
        "source": "Turbine Section Overhaul Manual — Section 4.2"
    },
    {
        "id": "doc_005",
        "text": "Oil system anomalies are early indicators of bearing failure in rotating machinery. Rising oil temperature, increased oil consumption, and metallic particles in oil samples indicate bearing wear. In turbofan engines, bearing failure can lead to catastrophic engine damage within 10-50 cycles. Recommended monitoring: oil debris analysis every 25 cycles, oil pressure differential checks, vibration signature analysis.",
        "source": "Engine Oil System Maintenance Guide — Section 2.1"
    },
    {
        "id": "doc_006",
        "text": "Predictive maintenance programs reduce unplanned downtime by 30-50% compared to reactive maintenance. Key performance indicators include Mean Time Between Failures (MTBF), Mean Time To Repair (MTTR), and Overall Equipment Effectiveness (OEE). For turbofan engines, a well-implemented predictive maintenance program can extend engine life by 15-20% and reduce maintenance costs by 25%. Data-driven approaches using machine learning models achieve RMSE below 15 cycles on standard benchmarks.",
        "source": "Predictive Maintenance ROI Study — AeroMaintenance Journal 2024"
    }
]

print(f"Documents prepared : {len(documents)}")
for doc in documents:
    print(f"  - {doc['id']} : {doc['source']}")

Documents prepared : 6
  - doc_001 : Turbofan Engine Maintenance Manual — Chapter 7
  - doc_002 : Turbofan Engine Maintenance Manual — Chapter 3
  - doc_003 : Predictive Maintenance Best Practices — ISO 13374
  - doc_004 : Turbine Section Overhaul Manual — Section 4.2
  - doc_005 : Engine Oil System Maintenance Guide — Section 2.1
  - doc_006 : Predictive Maintenance ROI Study — AeroMaintenance Journal 2024


In [4]:
# Load the embedding model
# all-MiniLM-L6-v2 is a lightweight but powerful model
# It transforms any text into a vector of 384 dimensions
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

# Transform each document into a vector and store in ChromaDB
print("\nIndexing documents...")

for doc in documents:
    # Transform text to vector — this is the embedding
    vector = embedding_model.encode(doc["text"]).tolist()
    
    # Store in ChromaDB : vector + original text + metadata
    collection.add(
        ids=[doc["id"]],
        embeddings=[vector],
        documents=[doc["text"]],
        metadatas=[{"source": doc["source"]}]
    )
    print(f"  ✓ Indexed : {doc['id']}")

print(f"\nTotal documents in collection : {collection.count()}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13678.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded

Indexing documents...
  ✓ Indexed : doc_001
  ✓ Indexed : doc_002
  ✓ Indexed : doc_003
  ✓ Indexed : doc_004
  ✓ Indexed : doc_005
  ✓ Indexed : doc_006

Total documents in collection : 6


In [5]:
# Test the retrieval — search for relevant documents given a query
query = "What should I do when HPC efficiency is degraded ?"

# Step 1 : transform the query into a vector
query_vector = embedding_model.encode(query).tolist()

# Step 2 : search for the 3 most similar documents in ChromaDB
results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

# Step 3 : display results
print(f"Query : {query}")
print(f"\nTop 3 most relevant documents :\n")

for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"Rank {i+1} — Score : {distance:.4f}")
    print(f"Source : {metadata['source']}")
    print(f"Content : {doc[:200]}...")
    print()

Query : What should I do when HPC efficiency is degraded ?

Top 3 most relevant documents :

Rank 1 — Score : 0.6330
Source : Turbofan Engine Maintenance Manual — Chapter 7
Content : High Pressure Compressor (HPC) degradation is one of the most common failure modes in turbofan engines. Symptoms include gradual decrease in HPC efficiency, rising exhaust gas temperature (EGT), and i...

Rank 2 — Score : 1.2539
Source : Turbine Section Overhaul Manual — Section 4.2
Content : Thermal stress in turbine sections occurs when operating temperatures exceed design limits. High Pressure Turbine (HPT) nozzle guide vanes are particularly vulnerable. Indicators include elevated sens...

Rank 3 — Score : 1.4001
Source : Turbofan Engine Maintenance Manual — Chapter 3
Content : Fan degradation in turbofan engines manifests as reduced fan speed (N1) and decreased bypass ratio. Physical fan speed readings above 9000 rpm combined with low corrected fan speed indicate fan effici...



In [6]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def rag_diagnosis(query: str, predicted_rul: float, n_results: int = 3) -> str:
    """
    Full RAG pipeline : retrieve relevant docs + generate diagnosis.
    
    Args:
        query: the maintenance question or sensor anomaly description
        predicted_rul: RUL predicted by the ML model
        n_results: number of documents to retrieve
    Returns:
        str: diagnosis grounded in technical documentation
    """
    # Step 1 — Retrieve relevant documents
    query_vector = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )
    
    # Step 2 — Build context from retrieved documents
    # Include source and relevance score for transparency
    context = ""
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        relevance = round((1 - distance) * 100, 1)
        context += f"\n[Document {i+1} — Source: {metadata['source']} — Relevance: {relevance}%]\n"
        context += doc + "\n"
    
    # Step 3 — Generate diagnosis with Claude
    prompt = f"""You are analyzing an industrial turbofan engine.

ML MODEL PREDICTION:
- Predicted RUL: {predicted_rul:.0f} cycles remaining
- Alert level: {"CRITICAL" if predicted_rul <= 20 else "WARNING" if predicted_rul <= 50 else "NORMAL"}

SENSOR ANOMALY DETECTED:
{query}

RELEVANT TECHNICAL DOCUMENTATION:
{context}

Based on the ML prediction AND the technical documentation above, provide:
1. Root cause analysis (what is likely failing)
2. Recommended actions with priority (P1/P2/P3)
3. Reference to the relevant documentation

Be precise and cite the documentation sources."""

    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=800,
        system="You are a senior predictive maintenance engineer. Always base your recommendations on the provided documentation. Respond in English.",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return message.content[0].text


# Test the full RAG pipeline
print("Testing RAG pipeline...\n")
result = rag_diagnosis(
    query="HPC efficiency sensor showing degradation, sensor_11 reading 47.91, sensor_9 elevated at 9065 rpm",
    predicted_rul=18.0
)
print(result)

Testing RAG pipeline...

# Predictive Maintenance Analysis Report

## Engine Status: CRITICAL — Immediate Action Required

---

## 1. Root Cause Analysis

Based on the sensor data and ML prediction, the primary failure mode is **High Pressure Compressor (HPC) degradation**.

**Evidence supporting this diagnosis:**

| Indicator | Current Reading | Threshold/Reference | Assessment |
|-----------|----------------|---------------------|------------|
| HPC efficiency (sensor_11) | 47.91 | Below 47% requires immediate inspection | **At threshold — borderline critical** |
| Physical fan speed (sensor_9) | 9,065 rpm | >9,000 rpm indicates efficiency loss | **Elevated — confirms degradation** |
| Predicted RUL | 18 cycles | Critical threshold | **Imminent failure risk** |

**Likely failure mechanism:** The combination of HPC efficiency approaching the critical 47% threshold with elevated fan speed suggests the compressor is working harder to compensate for internal degradation. Per Document 1, 